# Recherche du meilleur modèle en maximisant le recall pour générer le maximum de gain possible

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
from sklearn.metrics import precision_recall_curve

In [9]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    precision_recall_curve, confusion_matrix,
    classification_report, make_scorer
)

# --- Data ---
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test  = pd.read_csv('../6.Data/Yann_Process_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'
selected_features = [col for col in train.columns if col not in [target, id_col]]

X = train[selected_features]
y = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- SMOTE ici, avant tout entraînement ---
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

scale = (y_train_res == 0).sum() / (y_train_res == 1).sum()  # ~1.0 après SMOTE

# --- Modèle ---
model = XGBClassifier(
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr'
)

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3],
}

scorer = make_scorer(recall_score, pos_label=1)  # ← recall

grid = RandomizedSearchCV(
    model, param_grid,
    n_iter=20,
    cv=5,
    scoring=scorer,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
grid.fit(X_train_res, y_train_res)  # ← données rééquilibrées

best_model = grid.best_estimator_
print('Best params:', grid.best_params_)

# --- Seuil optimisé pour le recall avec précision minimum ---
y_proba = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

min_precision = 0.3  # ← ajuste selon ton contexte métier
valid = precisions[:-1] >= min_precision

if valid.sum() > 0:
    best_threshold = thresholds[valid][recalls[:-1][valid].argmax()]
else:
    best_threshold = 0.3  # fallback
    print("⚠️ Aucun seuil ne respecte la précision minimum, fallback à 0.3")

print(f"Seuil optimal : {best_threshold:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}
Seuil optimal : 0.209

Confusion Matrix:
[[25878  3367]
 [ 1312  1443]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.88      0.92     29245
           1       0.30      0.52      0.38      2755

    accuracy                           0.85     32000
   macro avg       0.63      0.70      0.65     32000
weighted avg       0.90      0.85      0.87     32000



In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    fbeta_score, precision_recall_curve,
    confusion_matrix, classification_report, make_scorer
)
from sklearn.calibration import CalibratedClassifierCV
from imblearn.combine import SMOTETomek

# ============================================================
# 1. IMPORT DES DONNÉES
# ============================================================
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test  = pd.read_csv('../6.Data/Yann_Process_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
def feature_engineering(df):
    df = df.copy()

    # Comportement transactionnel
    df['avg_amount_per_transaction'] = df['avg_amount_30d_eur'] / (df['num_transactions_30d'] + 1)
    df['max_to_avg_ratio']           = df['max_amount_30d_eur'] / (df['avg_amount_30d_eur'] + 1)
    df['transaction_intensity']      = df['num_transactions_30d'] / (df['tenure_months'] + 1)

    # Signaux de risque combinés
    df['risk_score']       = df['ip_risk_z'] + df['is_vpn'] * 2 + df['partner_risk_indicator']
    df['device_risk']      = df['num_devices_30d'] * df['ip_risk_z']
    df['trust_risk_ratio'] = df['device_trust_z'] / (df['ip_risk_z'] + 1e-9)

    # Historique négatif
    df['incident_score']  = (df['chargebacks_12m'] * 3 +
                              df['failed_payments_6m'] * 2 +
                              df['support_tickets_90d'])
    df['chargeback_rate'] = df['chargebacks_12m'] / (df['num_transactions_30d'] + 1)

    # Profil client
    df['income_per_age']      = df['annual_income_eur'] / (df['age'] + 1)
    df['credit_income_ratio'] = df['credit_score'] / (df['annual_income_eur'] + 1)
    df['is_new_account']      = (df['account_age_days'] < 90).astype(int)
    df['is_inactive']         = (df['days_since_last_login'] > 30).astype(int)

    # Combinaisons à fort signal fraude
    df['new_account_vpn']       = df['is_new_account'] * df['is_vpn']
    df['new_account_high_risk'] = df['is_new_account'] * df['risk_score']
    df['inactive_high_amount']  = df['is_inactive'] * df['max_amount_30d_eur']
    df['vpn_high_amount']       = df['is_vpn'] * df['max_amount_30d_eur']
    df['multi_device_vpn']      = df['num_devices_30d'] * df['is_vpn']

    # Signaux internes agrégés
    internal_cols = [f'internal_signal_{i}' for i in range(1, 9)]
    df['internal_signal_sum']  = df[internal_cols].sum(axis=1)
    df['internal_signal_mean'] = df[internal_cols].mean(axis=1)
    df['internal_signal_max']  = df[internal_cols].max(axis=1)
    df['internal_signal_std']  = df[internal_cols].std(axis=1)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

selected_features = [col for col in train.columns if col not in [target, id_col]]

X      = train[selected_features]
y      = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ============================================================
# 3. ISOLATION FOREST — anomaly score comme feature
# ============================================================
print("Isolation Forest...")
iso = IsolationForest(contamination=0.1, random_state=42, n_jobs=-1)
iso.fit(X_train)

X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train['anomaly_score'] = iso.decision_function(X_train)
X_val['anomaly_score']   = iso.decision_function(X_val)
X_test['anomaly_score']  = iso.decision_function(X_test)

# ============================================================
# 4. RÉÉQUILIBRAGE — SMOTETomek
# ============================================================
print("SMOTETomek...")
smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

scale = (y_train_res == 0).sum() / (y_train_res == 1).sum()
print(f"Distribution après SMOTETomek : {pd.Series(y_train_res).value_counts().to_dict()}")

# ============================================================
# 5. STACKING
# ============================================================
print("Stacking en cours (~20-30 min)...")

estimators = [
    ('lgbm', LGBMClassifier(
        scale_pos_weight=scale,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )),
    ('rf', RandomForestClassifier(
        class_weight='balanced',
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    )),
    ('xgb', XGBClassifier(
        scale_pos_weight=scale,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
        eval_metric='aucpr',
        verbosity=0
    ))
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3,        # ← cv=3 pour réduire le temps
    n_jobs=-1,
    passthrough=False
)

stack.fit(X_train_res, y_train_res)
print("Stacking terminé ✅")

# ============================================================
# 6. CALIBRATION
# ============================================================
print("Calibration...")
calibrated = CalibratedClassifierCV(stack, method='isotonic', cv='prefit')
calibrated.fit(X_val, y_val)

# ============================================================
# 7. OPTIMISATION DU SEUIL — F-beta (β=2)
# ============================================================
y_proba = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

beta          = 2  # ← recall pèse 2x plus, monte à 3 si besoin
f_beta_scores = (1 + beta**2) * (precisions * recalls) / (beta**2 * precisions + recalls + 1e-9)
best_idx      = f_beta_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

print(f"\n--- Seuil optimal F-beta (β={beta}) : {best_threshold:.3f} ---")
print(f"Précision : {precisions[best_idx]:.3f}")
print(f"Recall    : {recalls[best_idx]:.3f}")
print(f"F-beta    : {f_beta_scores[best_idx]:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

# ============================================================
# 8. PRÉDICTION SUR LE TEST
# ============================================================
y_test_proba = calibrated.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id': test[id_col],
    'target_is_fraud': y_test_pred
})
submission.to_csv('../6.Data/submission.csv', index=False)
print(f"\nSubmission sauvegardée ✅ — {y_test_pred.sum()} fraudes détectées")

Isolation Forest...
SMOTETomek...
Distribution après SMOTETomek : {0: 116874, 1: 116874}
Stacking en cours (~20-30 min)...
Stacking terminé ✅
Calibration...


c:\Users\natha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(



--- Seuil optimal F-beta (β=2) : 0.102 ---
Précision : 0.225
Recall    : 0.707
F-beta    : 0.495

Confusion Matrix:
[[22552  6693]
 [  808  1947]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.77      0.86     29245
           1       0.23      0.71      0.34      2755

    accuracy                           0.77     32000
   macro avg       0.60      0.74      0.60     32000
weighted avg       0.90      0.77      0.81     32000


Submission sauvegardée ✅ — 10675 fraudes détectées


In [16]:
# Prédiction binaire 0/1 pour Kaggle
X_test_sel = X_test[X_train.columns]  # ← même colonnes que l'entraînement (inclut anomaly_score)

y_test_proba     = calibrated.predict_proba(X_test_sel)[:, 1]
test_predictions = (y_test_proba >= best_threshold).astype(int)  # ← seuil optimisé

submission = pd.DataFrame({
    "customer_id": test["customer_id"],
    "target_is_fraud": test_predictions
})

submission.to_csv('../7.Submission/submission_rf_simple.csv', index=False)
print(f"Submission exportée ✅ — {test_predictions.sum()} fraudes détectées")

Submission exportée ✅ — 11777 fraudes détectées


In [12]:
# ============================================================
# 7. OPTIMISATION DU SEUIL — F-beta β=2 sans contrainte rigide
# ============================================================
y_proba = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

# Tester plusieurs beta et afficher les résultats
print("\n--- Exploration des seuils ---")
for beta in [1.0, 1.5, 2.0, 2.5]:
    f_beta_scores = (1 + beta**2) * (precisions * recalls) / (beta**2 * precisions + recalls + 1e-9)
    idx = f_beta_scores[:-1].argmax()
    print(f"β={beta} → seuil={thresholds[idx]:.3f} | précision={precisions[idx]:.3f} | recall={recalls[idx]:.3f} | F1={2*(precisions[idx]*recalls[idx])/(precisions[idx]+recalls[idx]+1e-9):.3f}")

# Choisir β=2 comme compromis recall/précision
beta          = 2.0
f_beta_scores = (1 + beta**2) * (precisions * recalls) / (beta**2 * precisions + recalls + 1e-9)
best_idx      = f_beta_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

print(f"\n✅ Seuil retenu (β={beta}) : {best_threshold:.3f}")
print(f"Précision : {precisions[best_idx]:.3f}")
print(f"Recall    : {recalls[best_idx]:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))


--- Exploration des seuils ---
β=1.0 → seuil=0.198 | précision=0.289 | recall=0.547 | F1=0.378
β=1.5 → seuil=0.143 | précision=0.261 | recall=0.619 | F1=0.367
β=2.0 → seuil=0.107 | précision=0.215 | recall=0.737 | F1=0.332
β=2.5 → seuil=0.078 | précision=0.199 | recall=0.778 | F1=0.317

✅ Seuil retenu (β=2.0) : 0.107
Précision : 0.215
Recall    : 0.737

Confusion Matrix:
[[21817  7428]
 [  725  2030]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.75      0.84     29245
           1       0.21      0.74      0.33      2755

    accuracy                           0.75     32000
   macro avg       0.59      0.74      0.59     32000
weighted avg       0.90      0.75      0.80     32000



In [17]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.calibration import CalibratedClassifierCV

# ============================================================
# 1. IMPORT DES DONNÉES
# ============================================================
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test  = pd.read_csv('../6.Data/Yann_Process_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
def feature_engineering(df):
    df = df.copy()

    df['avg_amount_per_transaction'] = df['avg_amount_30d_eur'] / (df['num_transactions_30d'] + 1)
    df['max_to_avg_ratio']           = df['max_amount_30d_eur'] / (df['avg_amount_30d_eur'] + 1)
    df['transaction_intensity']      = df['num_transactions_30d'] / (df['tenure_months'] + 1)

    df['risk_score']       = df['ip_risk_z'] + df['is_vpn'] * 2 + df['partner_risk_indicator']
    df['device_risk']      = df['num_devices_30d'] * df['ip_risk_z']
    df['trust_risk_ratio'] = df['device_trust_z'] / (df['ip_risk_z'] + 1e-9)

    df['incident_score']  = (df['chargebacks_12m'] * 3 +
                              df['failed_payments_6m'] * 2 +
                              df['support_tickets_90d'])
    df['chargeback_rate'] = df['chargebacks_12m'] / (df['num_transactions_30d'] + 1)

    df['income_per_age']      = df['annual_income_eur'] / (df['age'] + 1)
    df['credit_income_ratio'] = df['credit_score'] / (df['annual_income_eur'] + 1)
    df['is_new_account']      = (df['account_age_days'] < 90).astype(int)
    df['is_inactive']         = (df['days_since_last_login'] > 30).astype(int)

    df['new_account_vpn']       = df['is_new_account'] * df['is_vpn']
    df['new_account_high_risk'] = df['is_new_account'] * df['risk_score']
    df['inactive_high_amount']  = df['is_inactive'] * df['max_amount_30d_eur']
    df['vpn_high_amount']       = df['is_vpn'] * df['max_amount_30d_eur']
    df['multi_device_vpn']      = df['num_devices_30d'] * df['is_vpn']

    internal_cols = [f'internal_signal_{i}' for i in range(1, 9)]
    df['internal_signal_sum']  = df[internal_cols].sum(axis=1)
    df['internal_signal_mean'] = df[internal_cols].mean(axis=1)
    df['internal_signal_max']  = df[internal_cols].max(axis=1)
    df['internal_signal_std']  = df[internal_cols].std(axis=1)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

selected_features = [col for col in train.columns if col not in [target, id_col]]

X      = train[selected_features]
y      = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ============================================================
# 3. ISOLATION FOREST
# ============================================================
print("Isolation Forest...")
iso = IsolationForest(contamination=0.1, random_state=42, n_jobs=-1)
iso.fit(X_train)

X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train['anomaly_score'] = iso.decision_function(X_train)
X_val['anomaly_score']   = iso.decision_function(X_val)
X_test['anomaly_score']  = iso.decision_function(X_test)

# ============================================================
# 4. PAS DE SMOTE — scale_pos_weight gère le déséquilibre
# ============================================================
X_train_res, y_train_res = X_train, y_train
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight : {scale:.2f}")

# ============================================================
# 5. STACKING
# ============================================================
print("Stacking en cours...")

estimators = [
    ('lgbm', LGBMClassifier(
        scale_pos_weight=scale,
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,  # ← réduit les faux positifs
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )),
    ('rf', RandomForestClassifier(
        class_weight='balanced',
        n_estimators=300,
        min_samples_leaf=20,   # ← réduit les faux positifs
        n_jobs=-1,
        random_state=42
    )),
    ('xgb', XGBClassifier(
        scale_pos_weight=scale,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=10,   # ← réduit les faux positifs
        random_state=42,
        n_jobs=-1,
        eval_metric='aucpr',
        verbosity=0
    ))
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=3,
    n_jobs=-1,
    passthrough=False
)

stack.fit(X_train_res, y_train_res)
print("Stacking terminé ✅")

# ============================================================
# 6. CALIBRATION
# ============================================================
print("Calibration...")
calibrated = CalibratedClassifierCV(stack, method='isotonic', cv='prefit')
calibrated.fit(X_val, y_val)

# ============================================================
# 7. OPTIMISATION DU SEUIL — F1 pur
# ============================================================
y_proba = calibrated.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

f1_scores_thresh = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx         = f1_scores_thresh[:-1].argmax()
best_threshold   = thresholds[best_idx]

print(f"\nSeuil optimal F1 : {best_threshold:.3f}")
print(f"Précision : {precisions[best_idx]:.3f}")
print(f"Recall    : {recalls[best_idx]:.3f}")
print(f"F1        : {f1_scores_thresh[best_idx]:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

# ============================================================
# 8. EXPORT SUBMISSION
# ============================================================
X_test_sel       = X_test[X_train.columns]
y_test_proba     = calibrated.predict_proba(X_test_sel)[:, 1]
test_predictions = (y_test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id': test[id_col],
    'target_is_fraud': test_predictions
})
submission.to_csv('../7.Submission/submission_f1.csv', index=False)
print(f"\nSubmission sauvegardée ✅ — {test_predictions.sum()} fraudes détectées")

Isolation Forest...
Scale pos weight : 10.61
Stacking en cours...
Stacking terminé ✅
Calibration...


c:\Users\natha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(



Seuil optimal F1 : 0.209
Précision : 0.330
Recall    : 0.530
F1        : 0.407

Confusion Matrix:
[[26287  2958]
 [ 1295  1460]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.90      0.93     29245
           1       0.33      0.53      0.41      2755

    accuracy                           0.87     32000
   macro avg       0.64      0.71      0.67     32000
weighted avg       0.90      0.87      0.88     32000


Submission sauvegardée ✅ — 5510 fraudes détectées


In [19]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix,
    classification_report, f1_score
)
from imblearn.over_sampling import SMOTE
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# 1. IMPORT DES DONNÉES
# ============================================================
train = pd.read_csv('../6.Data/Yann_Process_train.csv')
test  = pd.read_csv('../6.Data/Yann_Process_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
def feature_engineering(df):
    df = df.copy()

    df['avg_amount_per_transaction'] = df['avg_amount_30d_eur'] / (df['num_transactions_30d'] + 1)
    df['max_to_avg_ratio']           = df['max_amount_30d_eur'] / (df['avg_amount_30d_eur'] + 1)
    df['transaction_intensity']      = df['num_transactions_30d'] / (df['tenure_months'] + 1)

    df['risk_score']       = df['ip_risk_z'] + df['is_vpn'] * 2 + df['partner_risk_indicator']
    df['device_risk']      = df['num_devices_30d'] * df['ip_risk_z']
    df['trust_risk_ratio'] = df['device_trust_z'] / (df['ip_risk_z'] + 1e-9)

    df['incident_score']  = (df['chargebacks_12m'] * 3 +
                              df['failed_payments_6m'] * 2 +
                              df['support_tickets_90d'])
    df['chargeback_rate'] = df['chargebacks_12m'] / (df['num_transactions_30d'] + 1)

    df['income_per_age']      = df['annual_income_eur'] / (df['age'] + 1)
    df['credit_income_ratio'] = df['credit_score'] / (df['annual_income_eur'] + 1)
    df['is_new_account']      = (df['account_age_days'] < 90).astype(int)
    df['is_inactive']         = (df['days_since_last_login'] > 30).astype(int)

    df['new_account_vpn']       = df['is_new_account'] * df['is_vpn']
    df['new_account_high_risk'] = df['is_new_account'] * df['risk_score']
    df['inactive_high_amount']  = df['is_inactive'] * df['max_amount_30d_eur']
    df['vpn_high_amount']       = df['is_vpn'] * df['max_amount_30d_eur']
    df['multi_device_vpn']      = df['num_devices_30d'] * df['is_vpn']

    internal_cols = [f'internal_signal_{i}' for i in range(1, 9)]
    df['internal_signal_sum']  = df[internal_cols].sum(axis=1)
    df['internal_signal_mean'] = df[internal_cols].mean(axis=1)
    df['internal_signal_max']  = df[internal_cols].max(axis=1)
    df['internal_signal_std']  = df[internal_cols].std(axis=1)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

selected_features = [col for col in train.columns if col not in [target, id_col]]

X      = train[selected_features]
y      = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ============================================================
# 3. ISOLATION FOREST
# ============================================================
print("Isolation Forest...")
iso = IsolationForest(contamination=0.1, random_state=42, n_jobs=-1)
iso.fit(X_train)

X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train['anomaly_score'] = iso.decision_function(X_train)
X_val['anomaly_score']   = iso.decision_function(X_val)
X_test['anomaly_score']  = iso.decision_function(X_test)

# ============================================================
# 4. SMOTE ratio contrôlé
# ============================================================
print("SMOTE...")
smote = SMOTE(sampling_strategy=0.4, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

scale = (y_train_res == 0).sum() / (y_train_res == 1).sum()
print(f"Scale après SMOTE : {scale:.2f}")

# ============================================================
# 5. OPTUNA — optimisation LightGBM (~8-12 min)
# ============================================================
print("Optuna en cours (~8-12 min)...")

def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 500),
        'max_depth'         : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 100),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples' : trial.suggest_int('min_child_samples', 10, 80),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight'  : scale,
        'random_state'      : 42,
        'n_jobs'            : -1,
        'verbose'           : -1
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train_res, y_train_res,
        cv=3,                                        # cv=3 pour la vitesse
        scoring='f1',
        n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)  # 50 trials suffisent

print(f"Meilleur F1 CV : {study.best_value:.4f}")
print(f"Meilleurs params : {study.best_params}")

# ============================================================
# 6. ENTRAÎNEMENT FINAL
# ============================================================
best_params = study.best_params
best_params.update({'scale_pos_weight': scale, 'random_state': 42, 'n_jobs': -1, 'verbose': -1})

best_model = LGBMClassifier(**best_params)
best_model.fit(X_train_res, y_train_res)

# ============================================================
# 7. OPTIMISATION DU SEUIL — coût métier
# ============================================================
y_proba = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

# Coût métier : FN=130€, FP=15€
cost_fn   = 130
cost_fp   = 15
n_fraud   = (y_val == 1).sum()
n_legit   = (y_val == 0).sum()

costs = []
for i, t in enumerate(thresholds):
    fn   = (1 - recalls[i]) * n_fraud
    fp   = recalls[i] / (precisions[i] + 1e-9) * (1 - precisions[i]) * n_fraud
    cost = fn * cost_fn + fp * cost_fp
    costs.append(cost)

best_cost_idx      = np.argmin(costs)
best_threshold_cost = thresholds[best_cost_idx]

# Comparer F1 pur vs coût métier
f1_scores_thresh = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_f1_idx      = f1_scores_thresh[:-1].argmax()
best_threshold_f1 = thresholds[best_f1_idx]

print("\n--- Seuil F1 pur ---")
print(f"Seuil={best_threshold_f1:.3f} | Précision={precisions[best_f1_idx]:.3f} | Recall={recalls[best_f1_idx]:.3f} | F1={f1_scores_thresh[best_f1_idx]:.3f}")

print("\n--- Seuil coût métier optimal ---")
print(f"Seuil={best_threshold_cost:.3f} | Précision={precisions[best_cost_idx]:.3f} | Recall={recalls[best_cost_idx]:.3f} | F1={f1_scores_thresh[best_cost_idx]:.3f}")

# Utiliser le seuil coût métier
best_threshold = best_threshold_cost
y_pred = (y_proba >= best_threshold).astype(int)

print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

# ============================================================
# 8. EXPORT SUBMISSION
# ============================================================
y_test_proba     = best_model.predict_proba(X_test)[:, 1]
test_predictions = (y_test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id'   : test[id_col],
    'target_is_fraud': test_predictions
})
submission.to_csv('../7.Submission/submission_lgbm_optuna.csv', index=False)
print(f"\nSubmission sauvegardée ✅ — {test_predictions.sum()} fraudes détectées")

Isolation Forest...
SMOTE...
Scale après SMOTE : 2.50
Optuna en cours (~8-12 min)...


Best trial: 22. Best value: 0.773037: 100%|██████████| 50/50 [15:11<00:00, 18.23s/it]


Meilleur F1 CV : 0.7730
Meilleurs params : {'n_estimators': 469, 'max_depth': 8, 'learning_rate': 0.09983230963844689, 'num_leaves': 100, 'subsample': 0.7876996820144377, 'colsample_bytree': 0.8346176554970621, 'min_child_samples': 27, 'reg_alpha': 0.005545639941752214, 'reg_lambda': 0.0012703092736032447}

--- Seuil F1 pur ---
Seuil=0.300 | Précision=0.311 | Recall=0.494 | F1=0.381

--- Seuil coût métier optimal ---
Seuil=0.145 | Précision=0.222 | Recall=0.689 | F1=0.336

Confusion Matrix:
[[22595  6650]
 [  857  1898]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.77      0.86     29245
           1       0.22      0.69      0.34      2755

    accuracy                           0.77     32000
   macro avg       0.59      0.73      0.60     32000
weighted avg       0.90      0.77      0.81     32000


Submission sauvegardée ✅ — 10678 fraudes détectées


In [26]:
df67 = pd.read_csv("../6.Data/kaggle_b2_fraud_train_v4.csv")
df67[df67['target_is_fraud'] == 0]

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_2,internal_signal_3,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,target_is_fraud
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544,20,60.92,80.16,4.9,...,2.77585,-0.35668,0.73256,-0.90929,0.38747,-0.21200,1.78510,1,NaN,0
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703,21,112.11,571.12,0.3,...,-0.65897,0.50084,0.23241,-1.16817,-0.82509,-0.78065,-1.25603,1,2.244073,0
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703,17,47.53,204.18,25.3,...,-1.95970,-1.68684,0.85302,1.28198,1.06118,0.99292,-0.05549,1,NaN,0
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,23065.56,594,13,99.95,734.09,12.8,...,0.32109,-1.51914,-0.27024,-0.61402,1.32524,0.78992,-0.96780,1,NaN,0
5,CUST_UGXYNE3HEO,ACC_ZQH0QJCPD6RK,38,0,27498.34,651,24,135.61,877.68,14.5,...,0.23069,-1.29838,0.18867,-0.67785,-0.72637,0.97861,0.68785,1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159994,CUST_GTGTGDVRXH,ACC_9JPLYWIZKBBF,49,16,16555.01,670,27,73.91,539.96,3.0,...,-0.30830,-0.80699,1.07001,-0.30598,0.72183,-1.81103,-0.03099,1,NaN,0
159995,CUST_I81IW5SVRQ,ACC_UPDTFTYTSM0A,56,0,34775.62,727,21,51.72,226.11,3.8,...,0.91351,-1.06238,-0.95945,-0.66641,-0.70669,-0.25664,-1.02678,1,NaN,0
159997,CUST_I0JS1GTS98,ACC_9JJ84W64Z7GX,30,2,41148.54,738,20,29.34,119.81,0.7,...,-1.56646,0.16181,0.12341,-1.06187,0.30498,-0.07834,0.33083,1,NaN,0
159998,CUST_L7GUCJ3TFY,ACC_NGFXDR7HW1ZS,56,6,15558.62,719,25,88.56,553.16,22.6,...,0.67588,0.14722,-0.40554,-0.11118,0.85848,1.69012,0.06194,1,-1.132891,0


# Nouveau preprocess et modèle peu eplicable pour compétition Kaggle

In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────
# 0. CHARGEMENT
# ─────────────────────────────────────────────────────────────────
train = pd.read_csv('../6.Data/kaggle_b2_fraud_train_v4.csv')
test  = pd.read_csv('../6.Data/kaggle_b2_fraud_test_v4.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

print(f"Train : {train.shape} | Test : {test.shape}")
print(f"Taux de fraude : {train[target].mean():.2%}")

# ─────────────────────────────────────────────────────────────────
# 1. SUPPRESSION — LEAKAGE & COLONNES INUTILES
# ─────────────────────────────────────────────────────────────────
# Leakage confirmé : ces colonnes sont produites APRÈS la détection
# de fraude → elles ne seraient pas disponibles au moment de la décision
leakage_cols = [
    'manual_review_result',       # résultat de revue post-fraude
    'post_event_status_code',     # "post_event" = après l'événement
    'chargeback_resolution_time_days',  # nécessite qu'un chargeback ait eu lieu
]

# Colonnes sans valeur prédictive
useless_cols = [
    'account_id',                 # identifiant pur, pas de signal
    'terms_accepted_flag',        # variance nulle (toujours = 1)
    'referrer_code',              # trop haute cardinalité, pas de signal direct
    'secondary_email',            # 92% NaN, signal marginal
    'postal_code',                # trop haute cardinalité, redondant avec city/region
    # Doublons de colonnes existantes → on garde les originaux
    'income_estimate_alt_eur',    # doublon de annual_income_eur
    'credit_score_norm',          # doublon de credit_score
    'income_log',                 # on va recalculer proprement
]

cols_to_drop = leakage_cols + useless_cols
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
test  = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

print(f"\nAprès suppression leakage/inutiles : {train.shape}")

# ─────────────────────────────────────────────────────────────────
# 2. SÉPARATION X / y AVANT TOUT AUTRE TRAITEMENT
# (évite tout leakage lié aux stats du train sur le test)
# ─────────────────────────────────────────────────────────────────
y_train = train[target].copy()
X_train = train.drop(columns=[target, id_col])
X_test  = test.drop(columns=[id_col])

# ─────────────────────────────────────────────────────────────────
# 3. NETTOYAGE DES VALEURS ABERRANTES AVANT IMPUTATION
# ─────────────────────────────────────────────────────────────────
def clean_before_impute(df):
    df = df.copy()

    # annual_income_eur : valeurs négatives = erreurs de saisie → NaN
    if 'annual_income_eur' in df.columns:
        df.loc[df['annual_income_eur'] < 0, 'annual_income_eur'] = np.nan

    # age : valeurs négatives ou > 120 → NaN
    if 'age' in df.columns:
        df.loc[(df['age'] < 0) | (df['age'] > 120), 'age'] = np.nan

    # tenure_months : valeurs négatives → NaN
    if 'tenure_months' in df.columns:
        df.loc[df['tenure_months'] < 0, 'tenure_months'] = np.nan

    return df

X_train = clean_before_impute(X_train)
X_test  = clean_before_impute(X_test)

# ─────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING (avant imputation pour utiliser les NaN)
# ─────────────────────────────────────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # ── Signup date ──────────────────────────────────────────────
    if 'signup_date' in df.columns:
        df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
        reference_date    = pd.Timestamp('2026-01-01')  # date fixe pour reproductibilité
        df['account_age_days']  = (reference_date - df['signup_date']).dt.days
        df['signup_month']      = df['signup_date'].dt.month   # saisonnalité
        df['signup_year']       = df['signup_date'].dt.year
        df['signup_is_weekend'] = df['signup_date'].dt.dayofweek.isin([5, 6]).astype(int)
        df = df.drop(columns=['signup_date'])

    # ── Variables textuelles (last_ticket_subject & customer_note) ─
    # Mots-clés à risque identifiés dans l'EDA
    risk_ticket_kw = ['suspicious', 'chargeback', 'dispute', 'locked', 'fraud',
                      'payment declined', 'refund']
    risk_note_kw   = ['suspicious login', 'address mismatch', 'refund twice',
                      'new device', 'high value order', 'shipping to new address',
                      'vpn', 'blocked']

    if 'last_ticket_subject' in df.columns:
        ticket_lower = df['last_ticket_subject'].fillna('').str.lower()
        df['ticket_is_risky'] = ticket_lower.str.contains(
            '|'.join(risk_ticket_kw), na=False).astype(int)
        # Encodage ordinal basique des sujets les plus fréquents
        ticket_risk_map = {
            'suspicious activity':  3,
            'chargeback dispute':   3,
            'payment declined':     2,
            'account locked':       2,
            'refund request':       2,
            'invoice issue':        1,
            'delivery problem':     1,
            'password reset':       1,
            'cancel subscription':  1,
            'change of address':    2,
        }
        df['ticket_risk_level'] = df['last_ticket_subject'].str.lower()\
            .map(ticket_risk_map).fillna(0)
        df = df.drop(columns=['last_ticket_subject'])

    if 'customer_note' in df.columns:
        note_lower = df['customer_note'].fillna('').str.lower()
        df['note_is_risky'] = note_lower.str.contains(
            '|'.join(risk_note_kw), na=False).astype(int)
        note_risk_map = {
            'user reports suspicious login': 3,
            'address mismatch detected':     3,
            'requested refund twice':        2,
            'shipping to new address':       2,
            'high value order':              1,
            'normal usage pattern':          0,
            'vip customer':                  0,
            'customer called support':       1,
        }
        df['note_risk_level'] = df['customer_note'].str.lower()\
            .map(note_risk_map).fillna(0)
        df = df.drop(columns=['customer_note'])

    # ── Interactions comportementales ─────────────────────────────
    # Signal fort : VPN + nouveau device simultanément
    if all(c in df.columns for c in ['is_vpn', 'is_new_device']):
        df['vpn_x_new_device'] = df['is_vpn'] * df['is_new_device']

    # Historique de risque combiné
    if all(c in df.columns for c in ['chargebacks_12m', 'failed_payments_6m']):
        df['total_risk_history'] = df['chargebacks_12m'] + df['failed_payments_6m']
        df['chargebacks_x_failed'] = df['chargebacks_12m'] * df['failed_payments_6m']

    # Montants anormalement élevés pour le niveau de revenu
    if all(c in df.columns for c in ['max_amount_30d_eur', 'annual_income_eur']):
        df['max_amount_to_income'] = df['max_amount_30d_eur'] / (df['annual_income_eur'].clip(lower=1) / 12)

    # Transaction inhabituelle : max très différent de la moyenne
    if all(c in df.columns for c in ['max_amount_30d_eur', 'avg_amount_30d_eur']):
        df['spike_ratio'] = df['max_amount_30d_eur'] / (df['avg_amount_30d_eur'].clip(lower=0.01))

    # Activité intense par rapport à l'ancienneté
    if all(c in df.columns for c in ['num_transactions_30d', 'tenure_months']):
        df['tx_per_tenure_month'] = df['num_transactions_30d'] / (df['tenure_months'].clip(lower=1))

    # Vélocité : transactions rapides après une longue inactivité
    if all(c in df.columns for c in ['num_transactions_30d', 'days_since_last_login']):
        df['tx_velocity'] = df['num_transactions_30d'] / (df['days_since_last_login'] + 1)

    # Score de risque device global
    if all(c in df.columns for c in ['device_trust_z', 'ip_risk_z']):
        df['device_risk_composite'] = df['ip_risk_z'] - df['device_trust_z']

    # Ratio tickets / transactions = comportement support anormal
    if all(c in df.columns for c in ['support_tickets_90d', 'num_transactions_30d']):
        df['tickets_per_tx'] = df['support_tickets_90d'] / (df['num_transactions_30d'] + 1)

    # Compte à faible ancienneté + revenu déclaré élevé = profil suspect
    if all(c in df.columns for c in ['annual_income_eur', 'tenure_months']):
        df['income_to_tenure'] = df['annual_income_eur'] / (df['tenure_months'].clip(lower=1))

    # ── Géographie ────────────────────────────────────────────────
    if 'country' in df.columns:
        df['is_foreign'] = (df['country'] != 'FR').astype(int)

    # ── legacy_partner_score : présence = signal en soi ───────────
    if 'legacy_partner_score' in df.columns:
        df['has_legacy_score'] = df['legacy_partner_score'].notna().astype(int)

    # ── partner_risk_indicator : présence = signal en soi ─────────
    if 'partner_risk_indicator' in df.columns:
        df['partner_risk_missing'] = df['partner_risk_indicator'].isna().astype(int)

    return df

X_train = feature_engineering(X_train)
X_test  = feature_engineering(X_test)

print(f"Après feature engineering : {X_train.shape[1]} features")

# ─────────────────────────────────────────────────────────────────
# 5. GESTION DES VALEURS MANQUANTES
# ─────────────────────────────────────────────────────────────────
# Colonnes numériques à imputer par la MÉDIANE (calculée sur train uniquement)
median_cols = [
    'age', 'tenure_months', 'annual_income_eur', 'credit_score',
    'num_transactions_30d', 'avg_amount_30d_eur', 'max_amount_30d_eur',
    'days_since_last_login', 'support_tickets_90d',
    'tx_amount_total_30d_eur', 'max_to_avg_ratio',
    'chargeback_resolution_time_days',
    # features engineerées dérivées de colonnes potentiellement manquantes
    'account_age_days', 'max_amount_to_income', 'income_to_tenure',
    'tx_per_tenure_month', 'tx_velocity', 'spike_ratio', 'tickets_per_tx',
]

# Colonnes numériques à imputer par 0 (z-scores et signaux = valeur neutre)
zero_cols = [
    'device_trust_z', 'ip_risk_z',
    'internal_signal_1', 'internal_signal_2', 'internal_signal_3',
    'internal_signal_4', 'internal_signal_5', 'internal_signal_6',
    'internal_signal_7', 'internal_signal_8',
    'legacy_partner_score', 'partner_risk_indicator',
    'chargebacks_12m', 'failed_payments_6m',
    'device_risk_composite', 'chargebacks_x_failed', 'total_risk_history',
]

# Calcul des médianes sur le train uniquement (pas de data leakage)
medians = {}
for col in median_cols:
    if col in X_train.columns:
        medians[col] = X_train[col].median()

for col, med in medians.items():
    X_train[col] = X_train[col].fillna(med)
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna(med)

for col in zero_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna(0)
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna(0)

# Colonnes catégorielles → imputation par 'unknown'
cat_cols_to_fill = ['channel', 'signup_source', 'plan_type', 'payment_method',
                    'browser', 'os', 'occupation', 'device_type',
                    'merchant_category', 'country', 'region', 'city']
for col in cat_cols_to_fill:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('unknown')
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna('unknown')

# ─────────────────────────────────────────────────────────────────
# 6. TRAITEMENT DES OUTLIERS (clipping sur percentiles du train)
# ─────────────────────────────────────────────────────────────────
# Clipping P1-P99 pour les skewed features
clip_p99_cols = [
    'annual_income_eur', 'avg_amount_30d_eur', 'max_amount_30d_eur',
    'tx_amount_total_30d_eur', 'num_transactions_30d',
    'days_since_last_login', 'support_tickets_90d',
    'max_amount_to_income', 'income_to_tenure', 'tx_velocity',
    'spike_ratio', 'tickets_per_tx',
]

clip_bounds = {}
for col in clip_p99_cols:
    if col in X_train.columns:
        lo = X_train[col].quantile(0.01)
        hi = X_train[col].quantile(0.99)
        clip_bounds[col] = (lo, hi)
        X_train[col] = X_train[col].clip(lo, hi)
        if col in X_test.columns:
            X_test[col] = X_test[col].clip(lo, hi)

# Clipping fixe pour les z-scores (par définition bornés à ±4/5)
zscore_cols = [
    'device_trust_z', 'ip_risk_z',
    'internal_signal_1', 'internal_signal_2', 'internal_signal_3',
    'internal_signal_4', 'internal_signal_5', 'internal_signal_6',
    'internal_signal_7', 'internal_signal_8',
]
for col in zscore_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].clip(-4, 4)
    if col in X_test.columns:
        X_test[col] = X_test[col].clip(-4, 4)

# ─────────────────────────────────────────────────────────────────
# 7. TRANSFORMATIONS NUMÉRIQUES
# ─────────────────────────────────────────────────────────────────
# Log-transform pour les distributions très skewées
log_cols = [
    'annual_income_eur', 'avg_amount_30d_eur', 'max_amount_30d_eur',
    'tx_amount_total_30d_eur', 'account_age_days',
]
for col in log_cols:
    if col in X_train.columns:
        new_col = f'{col}_log'
        X_train[new_col] = np.log1p(X_train[col].clip(lower=0))
        X_test[new_col]  = np.log1p(X_test[col].clip(lower=0))
        # On garde les deux (original et log) pour que le modèle choisisse
        # Si tu veux réduire le nb de features, drop les originaux :
        # X_train = X_train.drop(columns=[col])
        # X_test  = X_test.drop(columns=[col])

# ─────────────────────────────────────────────────────────────────
# 8. ENCODAGE DES VARIABLES CATÉGORIELLES
# ─────────────────────────────────────────────────────────────────
# On utilise le Target Encoding pour les catégorielles à haute cardinalité
# et le One-Hot pour les basses cardinalités
# IMPORTANT : Target Encoding calculé sur le train uniquement

from sklearn.model_selection import StratifiedKFold

def target_encode(X_tr, y_tr, X_te, col, n_splits=5, smoothing=10):
    """
    Target encoding avec cross-fitting (évite le leakage dans le train).
    smoothing : régularisation vers la moyenne globale si peu d'observations.
    """
    global_mean = y_tr.mean()
    X_tr_enc = X_tr[col].copy()
    X_te_enc = X_te[col].copy() if col in X_te.columns else pd.Series(index=X_te.index)

    # Encodage du train en out-of-fold pour éviter le leakage
    skf    = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof    = pd.Series(index=X_tr.index, dtype=float)

    for tr_idx, val_idx in skf.split(X_tr, y_tr):
        fold_map = (
            y_tr.iloc[tr_idx]
            .groupby(X_tr[col].iloc[tr_idx])
            .agg(['sum', 'count'])
        )
        fold_map['encoded'] = (
            (fold_map['sum'] + smoothing * global_mean) /
            (fold_map['count'] + smoothing)
        )
        oof.iloc[val_idx] = X_tr[col].iloc[val_idx].map(fold_map['encoded'])

    oof = oof.fillna(global_mean)

    # Encodage du test sur l'ensemble du train
    full_map = (
        y_tr.groupby(X_tr[col])
        .agg(['sum', 'count'])
    )
    full_map['encoded'] = (
        (full_map['sum'] + smoothing * global_mean) /
        (full_map['count'] + smoothing)
    )
    te_encoded = X_te[col].map(full_map['encoded']).fillna(global_mean) if col in X_te.columns else pd.Series(global_mean, index=X_te.index)

    return oof.rename(f'{col}_te'), te_encoded.rename(f'{col}_te')

# Colonnes à haute cardinalité → Target Encoding
high_card_cols = ['occupation', 'merchant_category', 'signup_source',
                  'country', 'region', 'city', 'payment_method',
                  'plan_type', 'channel', 'os', 'browser', 'device_type']

for col in high_card_cols:
    if col in X_train.columns:
        tr_enc, te_enc = target_encode(X_train, y_train, X_test, col)
        X_train[f'{col}_te'] = tr_enc.values
        X_test[f'{col}_te']  = te_enc.values
        # On drop les originaux une fois encodés
        X_train = X_train.drop(columns=[col])
        X_test  = X_test.drop(columns=[col])

print(f"Après encodage : {X_train.shape[1]} features")

# ─────────────────────────────────────────────────────────────────
# 9. VÉRIFICATION FINALE
# ─────────────────────────────────────────────────────────────────
# Vérifier qu'il ne reste pas de NaN
nan_train = X_train.isnull().sum()
nan_test  = X_test.isnull().sum()
if nan_train.any():
    print(f"\n⚠️  NaN restants dans X_train :")
    print(nan_train[nan_train > 0])
else:
    print("\n✅ Aucun NaN dans X_train")

if nan_test.any():
    print(f"⚠️  NaN restants dans X_test :")
    print(nan_test[nan_test > 0])
else:
    print("✅ Aucun NaN dans X_test")

# Vérifier l'alignement des colonnes
missing_in_test  = set(X_train.columns) - set(X_test.columns)
missing_in_train = set(X_test.columns) - set(X_train.columns)
if missing_in_test:
    print(f"\n⚠️  Colonnes dans train mais pas test : {missing_in_test}")
if missing_in_train:
    print(f"⚠️  Colonnes dans test mais pas train : {missing_in_train}")
if not missing_in_test and not missing_in_train:
    print("✅ Colonnes identiques entre train et test")

print(f"\n── Shape finale ──")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape} | Fraudes : {y_train.mean():.2%}")

# ─────────────────────────────────────────────────────────────────
# 10. SAUVEGARDE
# ─────────────────────────────────────────────────────────────────
X_train_out = X_train.copy()
X_train_out['customer_id']   = train[id_col].values
X_train_out[target]          = y_train.values

X_test_out = X_test.copy()
X_test_out['customer_id']    = test[id_col].values

X_train_out.to_csv('../6.Data/Nathan_Preprocessed_train.csv', index=False)
X_test_out.to_csv('../6.Data/Nathan_Preprocessed_test.csv',  index=False)

print("\n✅ Fichiers sauvegardés :")
print("  ../6.Data/Nathan_Preprocessed_train.csv")
print("  ../6.Data/Nathan_Preprocessed_test.csv")

# ─────────────────────────────────────────────────────────────────
# RÉCAP DES FEATURES FINALES
# ─────────────────────────────────────────────────────────────────
print(f"\n── Features finales ({X_train.shape[1]}) ──")
for col in sorted(X_train.columns):
    print(f"  {col}")

Train : (160000, 34) | Test : (40000, 33)
Taux de fraude : 8.61%

Après suppression leakage/inutiles : (160000, 30)
Après feature engineering : 43 features
Après encodage : 47 features

✅ Aucun NaN dans X_train
✅ Aucun NaN dans X_test
✅ Colonnes identiques entre train et test

── Shape finale ──
X_train : (160000, 47)
X_test  : (40000, 47)
y_train : (160000,) | Fraudes : 8.61%

✅ Fichiers sauvegardés :
  ../6.Data/Nathan_Preprocessed_train.csv
  ../6.Data/Nathan_Preprocessed_test.csv

── Features finales (47) ──
  account_age_days
  account_age_days_log
  age
  annual_income_eur
  annual_income_eur_log
  avg_amount_30d_eur
  avg_amount_30d_eur_log
  chargebacks_12m
  chargebacks_x_failed
  country_te
  credit_score
  days_since_last_login
  device_risk_composite
  device_trust_z
  failed_payments_6m
  income_to_tenure
  internal_signal_1
  internal_signal_2
  internal_signal_3
  internal_signal_4
  internal_signal_5
  internal_signal_6
  internal_signal_7
  internal_signal_8
  ip_risk_

In [18]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, fbeta_score, make_scorer
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# ─────────────────────────────────────────────
# 1. CHARGEMENT
# ─────────────────────────────────────────────
train = pd.read_csv('../6.Data/Nathan_Preprocessed_train.csv')
test  = pd.read_csv('../6.Data/Nathan_Preprocessed_test.csv')

target = 'target_is_fraud'
id_col = 'customer_id'

selected_features = [col for col in train.columns if col not in [target, id_col]]

X      = train[selected_features]
y      = train[target]
X_test = test[selected_features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ─────────────────────────────────────────────
# 2. SMOTE (sur train uniquement)
# ─────────────────────────────────────────────
smote = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Après SMOTE — Train : {X_train_sm.shape} | Fraudes : {y_train_sm.mean():.2%}")

# ─────────────────────────────────────────────
# 3. GRIDSEARCH optimisé recall
# ─────────────────────────────────────────────
scale = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight = scale,
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist',
    eval_metric      = 'aucpr',
)

param_grid = {
    'n_estimators':     [200, 400],
    'max_depth':        [3, 4, 5],
    'learning_rate':    [0.05, 0.1],
    'subsample':        [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 5, 10],
}

# F1 scorer : équilibre precision et recall
f1_scorer = make_scorer(fbeta_score, beta=1, pos_label=1)

grid = GridSearchCV(
    model, param_grid,
    cv      = 5,
    scoring = f1_scorer,
    n_jobs  = -1,
    verbose = 1,
)
grid.fit(X_train_sm, y_train_sm)

best_model = grid.best_estimator_
print(f'\nBest params : {grid.best_params_}')
print(f'Best CV F1  : {grid.best_score_:.4f}')

# ─────────────────────────────────────────────
# 4. OPTIMISATION DU SEUIL (F2)
# ─────────────────────────────────────────────
y_proba                         = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)

f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold = thresholds[f1_scores[:-1].argmax()]
print(f"\nMeilleur seuil (F1) : {best_threshold:.3f}")

y_pred = (y_proba >= best_threshold).astype(int)

# ─────────────────────────────────────────────
# 5. MÉTRIQUES
# ─────────────────────────────────────────────
print('\nConfusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=['Normal', 'Fraude']))

# ─────────────────────────────────────────────
# 6. SUBMISSION
# ─────────────────────────────────────────────
y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'customer_id':   test[id_col],
    target:          y_test_pred,
})
submission.to_csv('../7.Submission/submission_xgb_improved.csv', index=False)
print(f"\n✅ Submission sauvegardée — {y_test_pred.sum()} fraudes prédites ({y_test_pred.mean():.2%})")

Après SMOTE — Train : (152071, 47) | Fraudes : 23.08%
Fitting 5 folds for each of 144 candidates, totalling 720 fits

Best params : {'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 10, 'n_estimators': 400, 'subsample': 0.7}
Best CV F1  : 0.6703

Meilleur seuil (F1) : 0.689

Confusion Matrix:
[[26206  3039]
 [ 1211  1544]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.96      0.90      0.92     29245
      Fraude       0.34      0.56      0.42      2755

    accuracy                           0.87     32000
   macro avg       0.65      0.73      0.67     32000
weighted avg       0.90      0.87      0.88     32000


✅ Submission sauvegardée — 18620 fraudes prédites (46.55%)


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_t = (y_proba >= t).astype(int)
    f1   = f1_score(y_val, y_pred_t, pos_label=1)
    prec = precision_score(y_val, y_pred_t, pos_label=1)
    rec  = recall_score(y_val, y_pred_t, pos_label=1)
    print(f"Seuil {t:.2f} → F1: {f1:.4f} | Precision: {prec:.3f} | Recall: {rec:.3f}")

Seuil 0.30 → F1: 0.2883 | Precision: 0.172 | Recall: 0.885
Seuil 0.40 → F1: 0.3247 | Precision: 0.202 | Recall: 0.821
Seuil 0.50 → F1: 0.3621 | Precision: 0.239 | Recall: 0.751
Seuil 0.60 → F1: 0.3989 | Precision: 0.286 | Recall: 0.660
Seuil 0.70 → F1: 0.4191 | Precision: 0.342 | Recall: 0.541
Seuil 0.80 → F1: 0.4085 | Precision: 0.430 | Recall: 0.389


In [20]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# Option 1 — Calibration des probabilités (2 min)
# Le modèle XGB mal calibré peut produire des probas écrasées vers 0/1
calibrated = CalibratedClassifierCV(best_model, cv='prefit', method='isotonic')
calibrated.fit(X_val, y_val)

y_proba_cal = calibrated.predict_proba(X_val)[:, 1]

for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba_cal >= t).astype(int)
    f1   = f1_score(y_val, y_pred_t, pos_label=1)
    prec = precision_score(y_val, y_pred_t, pos_label=1)
    rec  = recall_score(y_val, y_pred_t, pos_label=1)
    print(f"Seuil {t:.2f} → F1: {f1:.4f} | Precision: {prec:.3f} | Recall: {rec:.3f}")

Seuil 0.30 → F1: 0.3982 | Precision: 0.466 | Recall: 0.347
Seuil 0.40 → F1: 0.3464 | Precision: 0.558 | Recall: 0.251
Seuil 0.50 → F1: 0.2680 | Precision: 0.645 | Recall: 0.169
Seuil 0.60 → F1: 0.1727 | Precision: 0.728 | Recall: 0.098
Seuil 0.70 → F1: 0.0944 | Precision: 0.821 | Recall: 0.050


In [21]:
# Refit sans SMOTE, juste scale_pos_weight
scale = (y_train == 0).sum() / (y_train == 1).sum()

params = best_model.get_params()
params['scale_pos_weight'] = (y_train == 0).sum() / (y_train == 1).sum()

model_no_smote = XGBClassifier(**params)
model_no_smote.fit(X_train, y_train)

y_proba_ns = model_no_smote.predict_proba(X_val)[:, 1]

for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_t = (y_proba_ns >= t).astype(int)
    f1   = f1_score(y_val, y_pred_t, pos_label=1)
    prec = precision_score(y_val, y_pred_t, pos_label=1)
    rec  = recall_score(y_val, y_pred_t, pos_label=1)
    print(f"Seuil {t:.2f} → F1: {f1:.4f} | Precision: {prec:.3f} | Recall: {rec:.3f}")

Seuil 0.30 → F1: 0.3167 | Precision: 0.194 | Recall: 0.858
Seuil 0.40 → F1: 0.3558 | Precision: 0.230 | Recall: 0.789
Seuil 0.50 → F1: 0.3888 | Precision: 0.268 | Recall: 0.706
Seuil 0.60 → F1: 0.4132 | Precision: 0.312 | Recall: 0.612
Seuil 0.70 → F1: 0.4302 | Precision: 0.374 | Recall: 0.507
Seuil 0.80 → F1: 0.4177 | Precision: 0.470 | Recall: 0.376


In [25]:
# Calibrer le seuil pour que le taux de fraudes prédit = taux réel
target_fraud_rate = y.mean()  # 8.61%

# Trouver le seuil qui donne ce taux sur le test
seuils = np.percentile(y_test_proba, np.linspace(0, 100, 1000))
for s in sorted(seuils, reverse=True):
    taux = (y_test_proba >= s).mean()
    if taux <= target_fraud_rate:
        seuil_calibre = s
        break

print(f"Seuil calibré : {seuil_calibre:.3f}")
print(f"Fraudes prédites : {(y_test_proba >= seuil_calibre).mean():.2%}")

# Nouvelle submission
y_test_pred_cal = (y_test_proba >= seuil_calibre).astype(int)
submission = pd.DataFrame({
    'customer_id': test[id_col],
    target:        y_test_pred_cal,
})
submission.to_csv('../7.Submission/submission_calibrated.csv', index=False)
print("✅ Submission calibrée sauvegardée")

Seuil calibré : 0.999
Fraudes prédites : 0.00%
✅ Submission calibrée sauvegardée


In [26]:
# Comparer les distributions de quelques features clés
for col in ['device_risk_composite', 'ip_risk_z', 'device_trust_z']:
    print(f"{col} — Train: {X_train[col].mean():.3f} | Test: {X_test[col].mean():.3f}")

device_risk_composite — Train: -0.004 | Test: 0.006
ip_risk_z — Train: 0.000 | Test: 0.001
device_trust_z — Train: 0.004 | Test: -0.005


In [35]:
params = best_model.get_params()
params['scale_pos_weight'] = (y_train == 0).sum() / (y_train == 1).sum()

model_final = XGBClassifier(**params)
model_final.fit(X_train, y_train)  # ← pas de SMOTE

y_test_proba_new = model_final.predict_proba(X_test)[:, 1]
y_val_proba_new  = model_final.predict_proba(X_val)[:, 1]

# Vérification distribution
print(f"Val  — moyenne proba : {y_val_proba_new.mean():.3f}")
print(f"Test — moyenne proba : {y_test_proba_new.mean():.3f}")
print(f"Fraudes prédites test : {(y_test_proba_new >= 0.5).mean():.2%}")

# Seuil optimal sur val
from sklearn.metrics import f1_score, precision_score, recall_score
for t in [0.3, 0.4, 0.5, 0.6,0.69, 0.7,0.71,0.72,0.73, 0.8]:
    y_pred_t = (y_val_proba_new >= t).astype(int)
    f1   = f1_score(y_val, y_pred_t, pos_label=1)
    prec = precision_score(y_val, y_pred_t, pos_label=1)
    rec  = recall_score(y_val, y_pred_t, pos_label=1)
    print(f"Seuil {t:.2f} → F1: {f1:.4f} | Precision: {prec:.3f} | Recall: {rec:.3f}")

Val  — moyenne proba : 0.289
Test — moyenne proba : 0.287
Fraudes prédites test : 22.34%
Seuil 0.30 → F1: 0.3167 | Precision: 0.194 | Recall: 0.858
Seuil 0.40 → F1: 0.3558 | Precision: 0.230 | Recall: 0.789
Seuil 0.50 → F1: 0.3888 | Precision: 0.268 | Recall: 0.706
Seuil 0.60 → F1: 0.4132 | Precision: 0.312 | Recall: 0.612
Seuil 0.69 → F1: 0.4294 | Precision: 0.367 | Recall: 0.518
Seuil 0.70 → F1: 0.4302 | Precision: 0.374 | Recall: 0.507
Seuil 0.71 → F1: 0.4318 | Precision: 0.382 | Recall: 0.497
Seuil 0.72 → F1: 0.4306 | Precision: 0.388 | Recall: 0.484
Seuil 0.73 → F1: 0.4283 | Precision: 0.393 | Recall: 0.470
Seuil 0.80 → F1: 0.4177 | Precision: 0.470 | Recall: 0.376


In [ ]:
y_test_pred_final = (y_test_proba_new >= 0.71).astype(int)

submission = pd.DataFrame({
    'customer_id': test[id_col],
    target:        y_test_pred_final,
})
submission.to_csv('../7.Submission/submission_final_07.csv', index=False)
print(f"✅ Submission sauvegardée — {y_test_pred_final.sum()} fraudes prédites ({y_test_pred_final.mean():.2%})")

✅ Submission sauvegardée — 4364 fraudes prédites (10.91%)


: 